In [4]:
import sys
import os

sys.path.append(
    os.path.abspath(".")
)


In [5]:
sys.path.append(
    os.path.abspath("..")
)


In [6]:
from src.datasets.brats_dataset import create_file_list
from src.config import DATA_DIR
from src.transforms.preprocessing import (
    train_transform,
    val_transform
)


In [7]:
files = create_file_list(DATA_DIR)

print(len(files))


369


In [8]:
from sklearn.model_selection import train_test_split

files = create_file_list(DATA_DIR)

# 70% train
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# 15% val, 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_files))
print("Val  :", len(val_files))
print("Test :", len(test_files))


Train: 258
Val  : 55
Test : 56


In [9]:
from monai.data import Dataset

train_ds = Dataset(
    data=train_files,
    transform=train_transform
)

val_ds = Dataset(
    data=val_files,
    transform=val_transform
)

test_ds = Dataset(
    data=test_files,
    transform=val_transform
)


In [10]:
from src.graphs.create_graph_dataset import create_graph_dataset


In [11]:
train_graphs = create_graph_dataset(train_ds)

val_graphs = create_graph_dataset(val_ds)

test_graphs = create_graph_dataset(test_ds)


  0%|          | 0/258 [00:00<?, ?it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  0%|          | 1/258 [00:06<28:37,  6.68s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 2/258 [00:10<20:48,  4.88s/it]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 3/258 [

In [12]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=1,
    shuffle=False
)


In [13]:
from train.train_cnn_gcn import train_cnn_gcn

model = train_cnn_gcn(
    train_loader,
    val_loader,
    epochs=300,
    lr=5e-4,
    hidden=64
)


Epoch 000 | Train Loss: 0.5598 | Train Acc: 0.9004 | Val Loss: 0.4153 | Val Acc: 0.9465
Epoch 010 | Train Loss: 0.4798 | Train Acc: 0.9322 | Val Loss: 0.3713 | Val Acc: 0.9718
Epoch 020 | Train Loss: 0.4704 | Train Acc: 0.9364 | Val Loss: 0.3668 | Val Acc: 0.9806
Epoch 030 | Train Loss: 0.4616 | Train Acc: 0.9380 | Val Loss: 0.3670 | Val Acc: 0.9723
Epoch 040 | Train Loss: 0.4563 | Train Acc: 0.9378 | Val Loss: 0.3628 | Val Acc: 0.9734
Epoch 050 | Train Loss: 0.4503 | Train Acc: 0.9406 | Val Loss: 0.3640 | Val Acc: 0.9718

Early stopping at epoch 58

Training Complete
Best Validation Accuracy: 0.9813


In [ ]:
from evaluate_cnn_gcn import evaluate_cnn_gcn


evaluate_cnn_gcn(
    model,
    test_loader
)


===== Evaluation Results =====
Accuracy  : 0.9817494836973528
Precision : 0.7458796554359742
Recall    : 0.5502471637862302
F1 Score  : 0.6182726450148566

Confusion Matrix:

[[869511     72   3198    178]
 [  1167   1047   1396    111]
 [  8225    298   6470    209]
 [   920    107    450   1465]]

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99    872959
           1       0.69      0.28      0.40      3721
           2       0.56      0.43      0.48     15202
           3       0.75      0.50      0.60      2942

    accuracy                           0.98    894824
   macro avg       0.75      0.55      0.62    894824
weighted avg       0.98      0.98      0.98    894824


===== Dice Scores =====
Class 0: 0.9921
Class 1: 0.3992
Class 2: 0.4844
Class 3: 0.5973

===== BraTS Region Dice Scores =====
WT (Whole Tumor)      : 0.6268
TC (Tumor Core)       : 0.5379
ET (Enhancing Tumor)  : 0.5973
